# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico
## SCADA-Core Automática — Grupo 04: Classificação e Seleção de Grãos por Visão Computacional

Neste notebook implementamos a arquitetura de **Base de Conhecimento Industrial** e o **Motor de Inferência (*Forward Chaining*)** para diagnóstico de causa raiz na planta de seleção de grãos. Estruturamos regras de produção em Cláusulas de Horn, mecanismos de priorização por severidade e geração automatizada de relatórios com Procedimentos Operacionais Padrão (POP).


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionários em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR" # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str      # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int      # 1 a 10 (10 = mais urgente)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def listar_catalogo(self) -> List[Dict[str, Any]]:
        return [
            {
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Diagnóstico)": r.consequente,
                "Procedimento POP": r.procedimento_pop
            }
            for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True)
        ]

class MotorInferenciaSCADA:
    def __init__(self, base_conhecimento: BaseConhecimentoSCADA):
        self.bc = base_conhecimento

    def executar_forward_chaining(self, memoria_trabalho: Dict[str, bool]) -> List[Dict[str, Any]]:
        """
        Algoritmo de Encadeamento Direto (Forward Chaining):
        Avalia as regras ativadas pelos fatos e deriva os diagnósticos com priorização.
        """
        fatos_ativos = {k for k, v in memoria_trabalho.items() if v is True}
        regras_disparadas = []
        
        for regra in sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True):
            if regra.antecedentes.issubset(fatos_ativos):
                regras_disparadas.append(regra)
                # Adiciona o consequente como novo fato derivado
                fatos_ativos.add(regra.consequente)
                
        relatorio = [
            {
                "Regra": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "Diagnóstico Causa Raiz": r.descricao_diagnostico,
                "Ação Corretiva IHM (POP)": r.procedimento_pop,
                "Tempo Limite": f"{r.tempo_resposta_max_s:.1f}s"
            }
            for r in regras_disparadas
        ]
        return relatorio

print("[OK] Arquitetura do Sistema Especialista inicializada com sucesso!")


## 1. Cadastro da Base de Conhecimento Oficial da Planta de Grãos
Registro das **Regras de Produção R01 a R12** cobrindo todos os subsistemas da planta (Recepção, Tração, Pesagem, Visão, Pneumática e Descarte).


In [ ]:
bc_graos = BaseConhecimentoSCADA()

# Subsistema A: Recepção e Alimentador Vibratório
bc_graos.adicionar_regra(
    'R01', ['c_ALIM', 'p_MOV201', 'NOT_p_NB101', 'VAZAO_NULA_FT301'], 'OBSTRUCAO_BOCAL_FUNIL',
    'Obstrução Mecânica no Bocal do Funil de Recepção', 'MÉDIA', 6, 10.0,
    'POP-REC-01: Interromper alimentador e desobstruir grade de entrada de grãos'
)
bc_graos.adicionar_regra(
    'R02', ['p_NB101', 'c_ALIM'], 'NIVEL_MINIMO_FUNIL',
    'Funil de Recepção no Nível Mínimo Operacional', 'BAIXA', 4, 30.0,
    'POP-REC-02: Solicitar reabastecimento de grãos brutos no silo de entrada'
)

# Subsistema B: Tração da Esteira
bc_graos.adicionar_regra(
    'R03', ['c_ESTEIRA', 'p_JI201', 'NOT_p_MOV201'], 'TRAVAMENTO_MECANICO_ESTEIRA',
    'Travamento Mecânico no Rolo ou Motor da Esteira com Sobrecarga Elétrica', 'CRÍTICA', 10, 1.0,
    'POP-ELE-01: Bloqueio LOTO imediato, desarmar disjuntor e inspecionar mancais'
)
bc_graos.adicionar_regra(
    'R04', ['c_ESTEIRA', 'NOT_p_JI201', 'NOT_p_MOV201'], 'FALHA_ENCODER_PATINAGEM',
    'Falha no Encoder de Velocidade ST-201 ou Patinagem Severa da Correia', 'ALTA', 8, 3.0,
    'POP-MEC-02: Inspecionar acoplamento do encoder incremental e esticar correia'
)

# Subsistema C: Pesagem Dinâmica
bc_graos.adicionar_regra(
    'R05', ['SOBRECARGA_MASSA_WT301'], 'SOBRECARGA_ESTEIRA',
    'Sobrecarga de Massa de Grãos na Seção de Pesagem WT-301', 'MÉDIA', 6, 5.0,
    'POP-BAL-01: Reduzir amplitude do alimentador vibratório na IHM'
)
bc_graos.adicionar_regra(
    'R06', ['NOT_c_ALIM', 'p_MOV201', 'MASSA_RESIDUAL_WT301'], 'DERIVA_ZERO_BALANCA',
    'Deriva de Zero ou Acúmulo de Pó na Célula de Carga WT-301', 'BAIXA', 3, 60.0,
    'POP-BAL-02: Executar rotina de auto-zero / tara da balança e limpeza de calha'
)

# Subsistema D: Visão Computacional e Qualidade
bc_graos.adicionar_regra(
    'R07', ['NOT_p_KSA401'], 'FALHA_COMUNICACAO_CAMERA',
    'Falha de Comunicação ou Travamento do Algoritmo da Câmera KSA-401', 'ALTA', 9, 2.0,
    'POP-VIS-01: Reinicializar serviço de visão e verificar cabeamento GigE/USB'
)
bc_graos.adicionar_regra(
    'R08', ['TAXA_REJEICAO_ALTA'], 'LOTE_CONTAMINADO',
    'Lote de Matéria-Prima com Alto Índice de Contaminação / Defeitos (>35%)', 'MÉDIA', 5, 15.0,
    'POP-QUAL-01: Notificar controle de qualidade e segregar fornecedor do lote'
)
bc_graos.adicionar_regra(
    'R09', ['p_KSA401', 'p_XS401', 'REJEICAO_ANOMALA_CONSECUTIVA'], 'LENTE_OBSTRUIDA',
    'Lente da Câmera Obstruída por Poeira ou Falha na Iluminação LED', 'ALTA', 8, 5.0,
    'POP-VIS-02: Limpar domo óptico da câmera e verificar iluminação de alta frequência'
)

# Subsistema E: Pneumática e Ejeção
bc_graos.adicionar_regra(
    'R10', ['p_PAL601'], 'QUEDA_PRESSAO_PNEUMATICA',
    'Falha Crítica no Suprimento de Ar Comprimido da Planta (PAL-601)', 'ALTA', 9, 2.0,
    'POP-PNEU-01: Verificar compressor principal, dreno de linha e vazamentos de ar'
)
bc_graos.adicionar_regra(
    'R11', ['c_FY603', 'NOT_p_PAL601', 'NOT_p_ZSH601'], 'FALHA_VALVULA_FY603',
    'Falha Eletromecânica / Queima de Bobina na Válvula Ejetora FY-603', 'CRÍTICA', 10, 1.0,
    'POP-PNEU-02: Testar sinal 24VDC no solenoide e substituir válvula de ejeção'
)

# Subsistema F: Coleta e Silo de Rejeito
bc_graos.adicionar_regra(
    'R12', ['p_NC703'], 'SILO_REJEITO_CHEIO',
    'Silo de Descarte Categoria C Saturado (Nível > 90% - Risco de Transbordo)', 'ALTA', 8, 10.0,
    'POP-COL-01: Substituir caçamba de rejeitos C e resetar alarme de nível'
)

motor_diagnostico = MotorInferenciaSCADA(bc_graos)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (GRUPO 04) ===")
print(formatar_tabela(bc_graos.listar_catalogo()))


## 2. Simulação de Cenários Industriais e Diagnóstico em Tempo Real
Validação do motor de inferência sob múltiplos cenários de falha na planta industrial.


In [ ]:
# Cenário 1: Queda de Pressão Pneumática (PAL-601 Ativo)
memoria_cenario_1 = {
    'c_ALIM': True,
    'p_MOV201': True,
    'p_PAL601': True, # Pressostato acusando pressão baixa
    'p_KSA401': True
}
diag_1 = motor_diagnostico.executar_forward_chaining(memoria_cenario_1)
print("\n" + "="*80)
print("=== CENÁRIO 1: DIAGNÓSTICO DE QUEDA DE PRESSÃO PNEUMÁTICA ===")
print(formatar_tabela(diag_1))
assert any(d['Regra'] == 'R10' for d in diag_1)

# Cenário 2: Travamento Mecânico da Esteira com Sobrecarga Elétrica
memoria_cenario_2 = {
    'c_ESTEIRA': True,
    'p_JI201': True,       # Relé térmico atuado
    'NOT_p_MOV201': True,  # Velocidade medida nula
    'p_KSA401': True
}
diag_2 = motor_diagnostico.executar_forward_chaining(memoria_cenario_2)
print("\n" + "="*80)
print("=== CENÁRIO 2: DIAGNÓSTICO DE TRAVAMENTO MECÂNICO (TRIP CRÍTICO) ===")
print(formatar_tabela(diag_2))
assert any(d['Regra'] == 'R03' for d in diag_2)

# Cenário 3: Falha Eletromecânica na Válvula Ejetora FY-603
memoria_cenario_3 = {
    'c_FY603': True,       # Comando de disparo enviado
    'NOT_p_PAL601': True,  # Pressão de ar normal
    'NOT_p_ZSH601': True   # Sensor magnético não confirmou avanço do atuador
}
diag_3 = motor_diagnostico.executar_forward_chaining(memoria_cenario_3)
print("\n" + "="*80)
print("=== CENÁRIO 3: DIAGNÓSTICO DE FALHA NA VÁLVULA EJETORA FY-603 ===")
print(formatar_tabela(diag_3))
assert any(d['Regra'] == 'R11' for d in diag_3)

# Cenário 4: Lente da Câmera Obstruída por Poeira
memoria_cenario_4 = {
    'p_KSA401': True,
    'p_XS401': True,
    'REJEICAO_ANOMALA_CONSECUTIVA': True,
    'TAXA_REJEICAO_ALTA': True
}
diag_4 = motor_diagnostico.executar_forward_chaining(memoria_cenario_4)
print("\n" + "="*80)
print("=== CENÁRIO 4: DIAGNÓSTICO DE LENTE OBSTRUÍDA / ILUMINAÇÃO ===")
print(formatar_tabela(diag_4))
assert any(d['Regra'] == 'R09' for d in diag_4)
assert any(d['Regra'] == 'R08' for d in diag_4)

print("\n[OK] Todos os cenários industriais foram diagnosticados com sucesso!")
